# Сравнение сэмплеров после восстановления baseline

## Тема и problem statement

Исследуем DDIM, DPM-Solver++ 2M и Heun на замороженной U-Net
google/ddpm-cifar10-32. Как различаются FID, время и изображения при одинаковом
бюджете 39 NFE и одном семействе расписания uniform_t?

## Что выполняет ноутбук

1. Загружает полный финальный ZIP baseline с компьютера.
2. Проверяет SHA256 и восстанавливает исходники, результаты и точный начальный шум.
3. Проверяет сохранённый baseline, запускает тесты и численную сверку с Diffusers.
4. Выполняет короткий pilot трёх методов.
5. Генерирует по 5000 изображений для каждого метода при 39 NFE.
6. Сохраняет полные резервные ZIP между блоками генерации.
7. Вычисляет FID, строит таблицу и сравнительную сетку, упаковывает отчёт.

Выполненный baseline DDIM-100 восстанавливается из файлов. Его генерация и FID
уже сохранены. Новые вызовы модели относятся к проверкам и новым экспериментам.
В новой сессии потребуются установка зависимостей, загрузка весов, CIFAR-10 и Inception.

## Что загрузить

Один файл с компьютера, около 120 MiB.

`t4_ddim100_baseline_v1_full_final_fid_000000-005000_0270fdbb.zip`

Копии с суффиксом (1) или (2) также подходят. Исходники находятся внутри source.zip,
точные шумы находятся внутри initial_noise.pt. Дополнительный ZIP проекта не требуется.

## Как выполнять

Выберите **Среда выполнения → Сменить среду выполнения → T4 GPU**.
Запускайте по одной ячейке. После каждой ячейки скачивания дождитесь появления
готового файла в загрузках браузера, затем запускайте следующую генерацию.
Рабочая папка /content временная. Последний полный ZIP каждого метода позволяет
восстановить сохранённый прогресс при совпадающем окружении.

Библиотечный baseline имеет clipping=True и начало t=990. Новая группа имеет
clipping=False и диапазон t=999…0. Ранжируем три собственных метода внутри новой
группы. FID baseline остаётся отдельным результатом с явно указанными условиями.

In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from pathlib import Path
import torch

assert torch.cuda.is_available(), 'Выберите GPU T4'
assert 'T4' in torch.cuda.get_device_name(0), 'Для сравнения выберите T4'
print('GPU', torch.cuda.get_device_name(0))
print('PyTorch', torch.__version__)
PROJECT = Path('/content/diffusion-sampling')
SOURCE_BUNDLE = Path('/content/restored-source.zip')

## 1. Восстановить baseline из полного ZIP

Следующая ячейка определяет функции восстановления на стандартной библиотеке Python.
Затем выберите один полный финальный архив baseline на компьютере.
Проверяются контрольные суммы, последовательности PNG и батчей, соответствие исходников.
Конфликтующие существующие файлы вызывают остановку с объяснением.

In [ ]:
"""Restore a full experiment export using only Python's standard library."""

from io import BytesIO
import hashlib
import json
from pathlib import Path, PurePosixPath
import shutil
import tempfile
from zipfile import ZipFile


def _relative_path(name):
    path = PurePosixPath(name)
    if not name or path.is_absolute() or ".." in path.parts or "\\" in name:
        raise ValueError(f"Unsafe archive path: {name}")
    return path


def _destination(root, name):
    path = root.joinpath(*_relative_path(name).parts)
    if not path.resolve().is_relative_to(root.resolve()):
        raise ValueError(f"Path leaves the project: {name}")
    return path


def _source_fingerprint(source):
    prefix = "diffusion-sampling/src/diffusion_sampling/"
    records = []
    for name in sorted(source.namelist()):
        relative = name.removeprefix(prefix)
        if name.startswith(prefix) and relative.endswith(".py") and "/" not in relative:
            records.append((relative, hashlib.sha256(source.read(name)).hexdigest()))
    if not records:
        raise ValueError("Source ZIP has no experiment package")
    return hashlib.sha256(json.dumps(records, sort_keys=True).encode()).hexdigest()


def restore_full_archive(archive_path, project, restore_source=False, noise_equal=None):
    """Verify a full ZIP and restore its run, exact noise and optional source.

    Existing runs must be identical to the uploaded snapshot. Conflicting files
    cause an error before restoration. Run directories are published atomically.
    """
    project = Path(project).resolve()
    with ZipFile(archive_path) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)):
            raise ValueError("ZIP contains duplicate entry names")
        for name in names:
            _relative_path(name)
        metadata = json.loads(archive.read("export_manifest.json"))
        hashes = metadata["files_sha256"]
        if set(names) != set(hashes) | {"export_manifest.json"}:
            raise ValueError("ZIP entries differ from the export manifest")
        for name, expected in hashes.items():
            if hashlib.sha256(archive.read(name)).hexdigest() != expected:
                raise ValueError(f"Checksum mismatch: {name}")
        if metadata["kind"] != "full" or metadata["start_image"] != 0:
            raise ValueError("Upload a full archive starting at image zero")
        manifest = json.loads(archive.read("run/manifest.json"))
        summary = json.loads(archive.read("run/summary.json"))
        experiment_id = manifest["experiment_id"]
        count = summary["generated_images"]
        if metadata["experiment_id"] != experiment_id or summary["experiment_id"] != experiment_id:
            raise ValueError("Experiment identifiers differ")
        if metadata["end_image_exclusive"] != count or metadata["images_in_archive"] != count:
            raise ValueError("Snapshot image counts differ")
        expected_images = {f"run/images/{index:08d}.png" for index in range(count)}
        if {name for name in names if name.startswith("run/images/")} != expected_images:
            raise ValueError("Snapshot PNG sequence has gaps or extra entries")
        config = manifest["config"]
        expected_batches = {
            f"run/batches/{index:08d}.pt"
            for index in range(0, count, config["batch_size"])
        }
        if {name for name in names if name.startswith("run/batches/")} != expected_batches:
            raise ValueError("Snapshot batch sequence has gaps or extra entries")
        run_relative = _relative_path(config["output_dir"])
        noise_relative = _relative_path(config["noise_path"])
        if run_relative.parts[0] != "results" or noise_relative.parts[0] != "data":
            raise ValueError("Expected results/ and data/ paths in the saved config")
        run_dir = _destination(project, run_relative.as_posix())
        noise_path = _destination(project, noise_relative.as_posix())
        noise_bytes = archive.read("initial_noise.pt")
        if noise_path.exists() and noise_path.read_bytes() != noise_bytes:
            if noise_equal is None or not noise_equal(noise_path, BytesIO(noise_bytes)):
                raise ValueError("An existing noise bank differs from the uploaded bank")
        source_bytes = archive.read("source.zip")
        source_files = {}
        with ZipFile(BytesIO(source_bytes)) as source:
            if len(source.namelist()) != len(set(source.namelist())):
                raise ValueError("Source ZIP contains duplicate entry names")
            if _source_fingerprint(source) != manifest["environment"]["source_sha256"]:
                raise ValueError("Source ZIP differs from the generation source hash")
            for entry in source.infolist():
                relative = _relative_path(entry.filename)
                if relative.parts[0] != "diffusion-sampling" or len(relative.parts) < 2:
                    raise ValueError("Unexpected source ZIP root")
                if not entry.is_dir():
                    source_files[PurePosixPath(*relative.parts[1:]).as_posix()] = source.read(entry)
        if restore_source:
            for name, data in source_files.items():
                path = _destination(project, name)
                if path.exists() and path.read_bytes() != data:
                    raise ValueError(f"Existing source differs: {name}")
        else:
            package = project / "src/diffusion_sampling"
            records = [(path.name, hashlib.sha256(path.read_bytes()).hexdigest())
                       for path in sorted(package.glob("*.py"))]
            fingerprint = hashlib.sha256(json.dumps(records, sort_keys=True).encode()).hexdigest()
            if fingerprint != manifest["environment"]["source_sha256"]:
                raise ValueError("Restore the matching project source before this run")
        run_files = {name.removeprefix("run/"): name for name in names if name.startswith("run/")}
        if run_dir.exists():
            existing = {path.relative_to(run_dir).as_posix()
                        for path in run_dir.rglob("*") if path.is_file()}
            if existing != set(run_files):
                raise ValueError("Existing run differs from the uploaded snapshot")
            for relative, name in run_files.items():
                if _destination(run_dir, relative).read_bytes() != archive.read(name):
                    raise ValueError(f"Existing run differs: {relative}")
        project.mkdir(parents=True, exist_ok=True)
        if restore_source:
            for name, data in source_files.items():
                path = _destination(project, name)
                path.parent.mkdir(parents=True, exist_ok=True)
                path.write_bytes(data)
        noise_path.parent.mkdir(parents=True, exist_ok=True)
        if not noise_path.exists():
            with tempfile.NamedTemporaryFile(dir=noise_path.parent, delete=False) as stream:
                temporary_noise = Path(stream.name)
                stream.write(noise_bytes)
            temporary_noise.replace(noise_path)
        if not run_dir.exists():
            run_dir.parent.mkdir(parents=True, exist_ok=True)
            staging = Path(tempfile.mkdtemp(prefix=".restore-", dir=run_dir.parent))
            try:
                for relative, name in run_files.items():
                    path = _destination(staging, relative)
                    path.parent.mkdir(parents=True, exist_ok=True)
                    path.write_bytes(archive.read(name))
                staging.replace(run_dir)
            finally:
                if staging.exists():
                    shutil.rmtree(staging)
    return {"run_dir": run_dir, "noise_path": noise_path, "manifest": manifest,
            "summary": summary, "source_bytes": source_bytes}

In [ ]:
from google.colab import files

uploaded = files.upload()
assert len(uploaded) == 1, 'Выберите один полный ZIP baseline'
uploaded_name, archive_bytes = next(iter(uploaded.items()))
BASELINE_ZIP = Path('/content/baseline_full.zip')
BASELINE_ZIP.write_bytes(archive_bytes)
del uploaded, archive_bytes
baseline = restore_full_archive(BASELINE_ZIP, PROJECT, restore_source=True)
assert baseline['manifest']['config']['sampler'] == 'diffusers_ddim'
assert baseline['summary']['generation_complete']
assert baseline['summary']['generated_images'] == 5000
assert baseline['summary']['baseline_metrics_complete']
SOURCE_BUNDLE.write_bytes(baseline['source_bytes'])
BASELINE_DIR = baseline['run_dir']
print('Загружен', uploaded_name)
print('Восстановлена папка', BASELINE_DIR)
print('Сохранённый FID', baseline['summary']['fid'])

## 2. Установить зависимости и проверить восстановленные данные

FID ниже читается из сохранённого отчёта Colab. Проверяем его связь с точным набором
PNG, согласованность метрик с батчами и хеш банка шумов. Полный численный пересчёт
FID относится к новым методам в следующих разделах.

In [ ]:
%pip install -q -e /content/diffusion-sampling
%cd /content/diffusion-sampling
import sys, json, math, shutil, subprocess
from io import BytesIO
from zipfile import ZIP_DEFLATED, ZipFile
from uuid import uuid4
from IPython.display import Image, display

os.environ.setdefault('TORCH_HOME', str(PROJECT / '.cache/torch'))
os.environ.setdefault('HF_HOME', str(PROJECT / '.cache/huggingface-home'))
sys.path.insert(0, str(PROJECT / 'src'))
from diffusion_sampling.config import load_config
from diffusion_sampling.experiment import load_shards
from diffusion_sampling.exports import CheckpointExporter
from diffusion_sampling.noise import load_noise
from diffusion_sampling.runtime import configure_runtime, environment
from diffusion_sampling.storage import file_sha256, object_sha256, tensor_sha256, write_json
from diffusion_sampling.study import create_sweep

configure_runtime(4)
manifest = baseline['manifest']
summary = baseline['summary']
shards = load_shards(BASELINE_DIR, manifest)
assert sum(s['end'] - s['start'] for s in shards) == 5000
assert math.isclose(sum(s['seconds'] for s in shards), summary['generation_seconds'], rel_tol=1e-12)
assert all(s['nfe'] == 100 for s in shards)
assert summary['sample_equivalent_model_evaluations'] == 500000
bank = load_noise(baseline['noise_path'])
assert bank['count'] == 5000 and bank['seed'] == 50000
assert bank['sha256'] == manifest['noise_sha256']
fid = json.loads((BASELINE_DIR / 'fid.json').read_text())
assert fid['fid'] == summary['fid'] and math.isfinite(fid['fid'])
assert fid['generated_count'] == 5000
assert fid['reference']['split'] == 'train' and fid['reference']['count'] == 50000
assert fid['experiment_id'] == manifest['experiment_id']
images = sorted((BASELINE_DIR / 'images').glob('*.png'))
assert object_sha256([(p.name, file_sha256(p)) for p in images]) == fid['generated_images_sha256']
print('Восстановление проверено. FID baseline', fid['fid'])
display(Image(filename=str(BASELINE_DIR / 'preview.png')))
del shards

## 3. Зафиксировать настройки исследования

Общий бюджет 39 NFE. DDIM и DPM-Solver++ используют 39 узлов, Heun использует 20.
Все методы получают один сохранённый банк из 5000 шумов. Batch size 64, float32,
uniform_t, clipping=False. Для всех новых запусков используем одно окружение T4.

STUDY_NAME обозначает отдельную группу. При изменении настроек или окружения
задайте новое имя. Полные резервные ZIP включают точный шум, поэтому их размер
больше delta-архивов. Для восстановления каждого метода достаточно его последнего ZIP.

In [ ]:
STUDY_NAME = 'study_uniform39_v1'
NFE_BUDGET = 39
BATCH_SIZE = 64
NUM_IMAGES = 5000
BLOCK_BATCHES = 16
DOWNLOAD_RESULTS = True
RESTORE_STUDY_BACKUPS = False
RUN_LONG_REFERENCE = False

STUDY_DIR = PROJECT / 'results' / STUDY_NAME
EXPORT_DIR = PROJECT / 'results/exports' / STUDY_NAME
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
SESSION_LOG = EXPORT_DIR / 'execution.log'
CURRENT_ENVIRONMENT = environment(torch.device('cuda'))
exporter = CheckpointExporter(PROJECT, EXPORT_DIR, source_bundle=SOURCE_BUNDLE)
source_noise_hash = bank['sha256']

def cli(*args):
    command = [sys.executable, '-m', 'diffusion_sampling', *map(str, args)]
    with SESSION_LOG.open('a') as log:
        with subprocess.Popen(command, cwd=PROJECT, text=True, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            returncode = process.wait()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)

def download_one(path):
    if path is None:
        return
    print('Готовый ZIP', path, f'{path.stat().st_size / 2**20:.1f} MiB')
    if DOWNLOAD_RESULTS:
        files.download(str(path))
    print('Проверьте завершение скачивания в браузере перед следующей генерацией.')

def snapshot(config, stage):
    run_dir = PROJECT / config.output_dir
    logs = run_dir / 'logs'
    logs.mkdir(exist_ok=True)
    for name in ['tests.log', 'execution.log']:
        if (EXPORT_DIR / name).exists():
            shutil.copyfile(EXPORT_DIR / name, logs / name)
    return exporter.full(run_dir, stage=stage)

cli('doctor')

## 4. Продолжение ранее начатого исследования

При первом запуске оставьте RESTORE_STUDY_BACKUPS=False.
После обрыва сессии сначала снова загрузите baseline, затем поставьте True и выберите
последний полный ZIP каждого начатого метода этой группы. Можно выбрать до трёх файлов.
У законченных методов последующие ячейки проверят сохранённые данные.
При изменении окружения восстановленные измерения сохраняются отдельно. Для новой
сопоставимой группы потребуется новое STUDY_NAME и новые прогоны её методов.

In [ ]:
if RESTORE_STUDY_BACKUPS:
    uploaded_backups = files.upload()
    assert 1 <= len(uploaded_backups) <= 3, 'Выберите последний полный ZIP каждого метода'
    for name, data in uploaded_backups.items():
        with ZipFile(BytesIO(data)) as archive:
            saved_manifest = json.loads(archive.read('run/manifest.json'))
        assert saved_manifest['config']['output_dir'].startswith(f'results/{STUDY_NAME}/')
        assert saved_manifest['config']['num_samples'] == NUM_IMAGES
        assert saved_manifest['noise_sha256'] == source_noise_hash
        assert saved_manifest['environment'] == CURRENT_ENVIRONMENT, (
            'Окружение изменилось. Для новой группы задайте новое STUDY_NAME, '
            'отключите восстановление и выполните её методы заново.'
        )
        path = Path('/content') / Path(name).name
        path.write_bytes(data)
        restored = restore_full_archive(
            path, PROJECT,
            noise_equal=lambda left, right: (
                load_noise(left)['sha256'] == load_noise(right)['sha256'] == source_noise_hash
            ),
        )
        restored_shards = load_shards(restored['run_dir'], restored['manifest'])
        assert sum(s['end'] - s['start'] for s in restored_shards) == restored['summary']['generated_images']
        print('Восстановлено', restored['run_dir'], restored['summary']['generated_images'])
    del uploaded_backups, restored_shards
else:
    print('Новая группа экспериментов')

## 5. Тесты и численная сверка с Diffusers

Сначала выполняются тесты сохранённой версии исходников. Затем два коротких примера
на настоящей модели проверяют промежуточные состояния и фактическое число вызовов.
Веса U-Net загрузятся при первом обращении. Дождитесь успешного завершения.

In [ ]:
tests = subprocess.run(
    [sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
    cwd=PROJECT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
(EXPORT_DIR / 'tests.log').write_text(tests.stdout)
print(tests.stdout)
tests.check_returncode()
verification = subprocess.run(
    [sys.executable, 'scripts/verify_samplers.py', '--device', 'cuda',
     '--output', f'results/{STUDY_NAME}/verification.json'],
    cwd=PROJECT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
(EXPORT_DIR / 'verification.log').write_text(verification.stdout)
print(verification.stdout)
verification.check_returncode()
verification_report = json.loads((STUDY_DIR / 'verification.json').read_text())
assert verification_report['passed']

## 6. Короткий pilot на 16 сохранённых шумах

Три метода при 19 NFE. Используем первые 16 шумов восстановленного банка fid.
Проверяем генерацию и соответствие колонок сетки одному начальному шуму.
Качество полной выборки оценивается в следующих разделах через FID.

In [ ]:
pilot_configs = PROJECT / 'configs' / f'{STUDY_NAME}_pilot'
create_sweep(pilot_configs, name=f'{STUDY_NAME}_pilot', budgets=(19,),
             schedules=('uniform_t',), num_samples=16, batch_size=8,
             purpose='pilot', device='cuda',
             noise_path=manifest['config']['noise_path'], noise_seed=50000)
pilot_runs = []
for method in ['ddim', 'dpmpp_2m', 'heun']:
    config_path = pilot_configs / f'{method}_uniform_t_nfe19.json'
    cli('run', '--config', config_path)
    pilot_runs.append(PROJECT / load_config(config_path).output_dir)
cli('compare', '--runs', *pilot_runs, '--output', STUDY_DIR / 'pilot_comparison.json')
display(Image(filename=str(STUDY_DIR / 'pilot_comparison.png')))

## 7. Подготовить три полных запуска

Одна группа содержит три метода при 39 NFE, uniform_t, по 5000 изображений.
Ниже для каждого метода есть первый батч, пять блоков продолжения и отдельный FID.
После каждого блока создаётся полный ZIP. Скачивание выполняется следующей короткой
ячейкой. Дождитесь сохранения файла перед продолжением.

In [ ]:
CONFIG_DIR = PROJECT / 'configs' / STUDY_NAME
create_sweep(CONFIG_DIR, name=STUDY_NAME, budgets=(NFE_BUDGET,),
             schedules=('uniform_t',), num_samples=NUM_IMAGES, batch_size=BATCH_SIZE,
             purpose='baseline', device='cuda',
             noise_path=manifest['config']['noise_path'], noise_seed=50000)

def run_block(config, max_batches, stage):
    config_path = CONFIG_DIR / f'{config.sampler}_uniform_t_nfe{NFE_BUDGET}.json'
    run_dir = PROJECT / config.output_dir
    if (run_dir / 'summary.json').exists():
        previous = json.loads((run_dir / 'summary.json').read_text())
        if previous['generation_complete']:
            saved_manifest = json.loads((run_dir / 'manifest.json').read_text())
            assert saved_manifest['config'] == config.to_dict()
            assert saved_manifest['environment'] == CURRENT_ENVIRONMENT
            assert saved_manifest['noise_sha256'] == source_noise_hash
            saved_shards = load_shards(run_dir, saved_manifest)
            assert sum(s['end'] - s['start'] for s in saved_shards) == NUM_IMAGES
            print('Генерация уже завершена', config.sampler)
            return None
    cli('run', '--config', config_path, '--max-batches', max_batches)
    saved_manifest = json.loads((run_dir / 'manifest.json').read_text())
    result = json.loads((run_dir / 'summary.json').read_text())
    assert saved_manifest['noise_sha256'] == source_noise_hash
    assert result['nfe_per_image'] == NFE_BUDGET
    assert saved_manifest['environment'] == CURRENT_ENVIRONMENT
    print('Изображений', result['generated_images'], '/', NUM_IMAGES)
    print('Секунд на изображение', result['seconds_per_image'])
    print('Осталось минут денойзинга примерно',
          (NUM_IMAGES - result['generated_images']) * result['seconds_per_image'] / 60)
    return snapshot(config, stage)

def finish_method(config, visual_notes):
    run_dir = PROJECT / config.output_dir
    result = json.loads((run_dir / 'summary.json').read_text())
    assert result['generation_complete'] and result['generated_images'] == NUM_IMAGES
    if not result['baseline_metrics_complete']:
        cli('evaluate', '--run', run_dir, '--device', 'cuda', '--download')
    result = json.loads((run_dir / 'summary.json').read_text())
    assert result['baseline_metrics_complete'] and math.isfinite(result['fid'])
    if visual_notes.strip():
        (run_dir / 'visual_review.md').write_text(visual_notes.strip() + '\n')
    print('FID', result['fid'])
    print('NFE', result['nfe_per_image'])
    print('Время денойзинга', result['generation_seconds'])
    return snapshot(config, 'final_fid')

## DDIM. Первый батч и оценка времени

При восстановленном запуске читаются уже сохранённые батчи и добавляется максимум один.
Первый батч нового запуска содержит 64 изображения.

In [ ]:
config = load_config(CONFIG_DIR / f'ddim_uniform_t_nfe{NFE_BUDGET}.json')
latest_archive = run_block(config, 1, 'first_batch')
display(Image(filename=str(PROJECT / config.output_dir / 'preview.png')))

In [ ]:
download_one(latest_archive)

### DDIM. Блок 1 из 5

Добавляем до 16 батчей. Если генерация уже завершена, проверяются сохранённые результаты.

In [ ]:
latest_archive = run_block(config, BLOCK_BATCHES, 'block_1')

In [ ]:
download_one(latest_archive)

### DDIM. Блок 2 из 5

Добавляем до 16 батчей. Если генерация уже завершена, проверяются сохранённые результаты.

In [ ]:
latest_archive = run_block(config, BLOCK_BATCHES, 'block_2')

In [ ]:
download_one(latest_archive)

### DDIM. Блок 3 из 5

Добавляем до 16 батчей. Если генерация уже завершена, проверяются сохранённые результаты.

In [ ]:
latest_archive = run_block(config, BLOCK_BATCHES, 'block_3')

In [ ]:
download_one(latest_archive)

### DDIM. Блок 4 из 5

Добавляем до 16 батчей. Если генерация уже завершена, проверяются сохранённые результаты.

In [ ]:
latest_archive = run_block(config, BLOCK_BATCHES, 'block_4')

In [ ]:
download_one(latest_archive)

### DDIM. Блок 5 из 5

Добавляем до 16 батчей. Если генерация уже завершена, проверяются сохранённые результаты.

In [ ]:
latest_archive = run_block(config, BLOCK_BATCHES, 'block_5')

In [ ]:
download_one(latest_archive)

### DDIM. FID, визуальные наблюдения и полный финальный ZIP

Посмотрите сетку и запишите наблюдения в VISUAL_NOTES. При пустой строке текстовый
файл пропускается. FID сравнивает 5000 generated с 50000 CIFAR-10 train.
Уже сохранённый завершённый FID читается из отчёта. Для нового расчёта нужны CIFAR-10
и веса Inception. После выполнения скачайте полный ZIP следующей ячейкой.

In [ ]:
VISUAL_NOTES = ''
latest_archive = finish_method(config, VISUAL_NOTES)
display(Image(filename=str(PROJECT / config.output_dir / 'preview.png')))

In [ ]:
download_one(latest_archive)

## DPM-Solver++ 2M. Первый батч и оценка времени

При восстановленном запуске читаются уже сохранённые батчи и добавляется максимум один.
Первый батч нового запуска содержит 64 изображения.

In [ ]:
config = load_config(CONFIG_DIR / f'dpmpp_2m_uniform_t_nfe{NFE_BUDGET}.json')
latest_archive = run_block(config, 1, 'first_batch')
display(Image(filename=str(PROJECT / config.output_dir / 'preview.png')))

In [ ]:
download_one(latest_archive)

### DPM-Solver++ 2M. Блок 1 из 5

Добавляем до 16 батчей. Если генерация уже завершена, проверяются сохранённые результаты.

In [ ]:
latest_archive = run_block(config, BLOCK_BATCHES, 'block_1')

In [ ]:
download_one(latest_archive)

### DPM-Solver++ 2M. Блок 2 из 5

Добавляем до 16 батчей. Если генерация уже завершена, проверяются сохранённые результаты.

In [ ]:
latest_archive = run_block(config, BLOCK_BATCHES, 'block_2')

In [ ]:
download_one(latest_archive)

### DPM-Solver++ 2M. Блок 3 из 5

Добавляем до 16 батчей. Если генерация уже завершена, проверяются сохранённые результаты.

In [ ]:
latest_archive = run_block(config, BLOCK_BATCHES, 'block_3')

In [ ]:
download_one(latest_archive)

### DPM-Solver++ 2M. Блок 4 из 5

Добавляем до 16 батчей. Если генерация уже завершена, проверяются сохранённые результаты.

In [ ]:
latest_archive = run_block(config, BLOCK_BATCHES, 'block_4')

In [ ]:
download_one(latest_archive)

### DPM-Solver++ 2M. Блок 5 из 5

Добавляем до 16 батчей. Если генерация уже завершена, проверяются сохранённые результаты.

In [ ]:
latest_archive = run_block(config, BLOCK_BATCHES, 'block_5')

In [ ]:
download_one(latest_archive)

### DPM-Solver++ 2M. FID, визуальные наблюдения и полный финальный ZIP

Посмотрите сетку и запишите наблюдения в VISUAL_NOTES. При пустой строке текстовый
файл пропускается. FID сравнивает 5000 generated с 50000 CIFAR-10 train.
Уже сохранённый завершённый FID читается из отчёта. Для нового расчёта нужны CIFAR-10
и веса Inception. После выполнения скачайте полный ZIP следующей ячейкой.

In [ ]:
VISUAL_NOTES = ''
latest_archive = finish_method(config, VISUAL_NOTES)
display(Image(filename=str(PROJECT / config.output_dir / 'preview.png')))

In [ ]:
download_one(latest_archive)

## Heun. Первый батч и оценка времени

При восстановленном запуске читаются уже сохранённые батчи и добавляется максимум один.
Первый батч нового запуска содержит 64 изображения.

In [ ]:
config = load_config(CONFIG_DIR / f'heun_uniform_t_nfe{NFE_BUDGET}.json')
latest_archive = run_block(config, 1, 'first_batch')
display(Image(filename=str(PROJECT / config.output_dir / 'preview.png')))

In [ ]:
download_one(latest_archive)

### Heun. Блок 1 из 5

Добавляем до 16 батчей. Если генерация уже завершена, проверяются сохранённые результаты.

In [ ]:
latest_archive = run_block(config, BLOCK_BATCHES, 'block_1')

In [ ]:
download_one(latest_archive)

### Heun. Блок 2 из 5

Добавляем до 16 батчей. Если генерация уже завершена, проверяются сохранённые результаты.

In [ ]:
latest_archive = run_block(config, BLOCK_BATCHES, 'block_2')

In [ ]:
download_one(latest_archive)

### Heun. Блок 3 из 5

Добавляем до 16 батчей. Если генерация уже завершена, проверяются сохранённые результаты.

In [ ]:
latest_archive = run_block(config, BLOCK_BATCHES, 'block_3')

In [ ]:
download_one(latest_archive)

### Heun. Блок 4 из 5

Добавляем до 16 батчей. Если генерация уже завершена, проверяются сохранённые результаты.

In [ ]:
latest_archive = run_block(config, BLOCK_BATCHES, 'block_4')

In [ ]:
download_one(latest_archive)

### Heun. Блок 5 из 5

Добавляем до 16 батчей. Если генерация уже завершена, проверяются сохранённые результаты.

In [ ]:
latest_archive = run_block(config, BLOCK_BATCHES, 'block_5')

In [ ]:
download_one(latest_archive)

### Heun. FID, визуальные наблюдения и полный финальный ZIP

Посмотрите сетку и запишите наблюдения в VISUAL_NOTES. При пустой строке текстовый
файл пропускается. FID сравнивает 5000 generated с 50000 CIFAR-10 train.
Уже сохранённый завершённый FID читается из отчёта. Для нового расчёта нужны CIFAR-10
и веса Inception. После выполнения скачайте полный ZIP следующей ячейкой.

In [ ]:
VISUAL_NOTES = ''
latest_archive = finish_method(config, VISUAL_NOTES)
display(Image(filename=str(PROJECT / config.output_dir / 'preview.png')))

In [ ]:
download_one(latest_archive)

## 8. Сравнительная таблица и изображения

Проверяются одинаковые веса, шумы, точность, clipping, крайние точки, окружение,
batch size и протокол FID. Все три запуска должны быть завершены.
В строках сетки расположены методы, в колонках используется одинаковый шум.
Меньшее значение FID означает лучший результат по выбранной метрике и протоколу.
Один общий банк даёт первичное сравнение. Для оценки устойчивости нужны независимые банки.

In [ ]:
runs = [PROJECT / load_config(CONFIG_DIR / f'{method}_uniform_t_nfe{NFE_BUDGET}.json').output_dir
        for method in ['ddim', 'dpmpp_2m', 'heun']]
cli('compare', '--runs', *runs, '--output', STUDY_DIR / 'comparison.json')
report = json.loads((STUDY_DIR / 'comparison.json').read_text())
assert len(report['rows']) == 3 and report['budgets'] == [NFE_BUDGET]
for row in report['rows']:
    assert row['fid'] is not None
    print(row['sampler'], 'NFE', row['nfe'], 'FID', row['fid'],
          'сек/изображение', row['seconds_per_image'])
display(Image(filename=str(STUDY_DIR / 'comparison.png')))

## 9. Дополнительный длинный эталон и MSE

По умолчанию RUN_LONG_REFERENCE=False. При True выполняются DDIM-500 и DDIM-1000
на тех же 16 шумах pilot, проверяется разница между эталонами, вычисляется MSE
коротких методов к DDIM-1000. MSE считается по float32 до преобразования в PNG.
Это дополнительная ось оценки численной точности. Связь с FID проверяется отдельно.
Подбор расписания и проверка на новых независимых шумах относятся к дальнейшему этапу.

In [ ]:
if RUN_LONG_REFERENCE:
    from dataclasses import replace
    references = []
    for budget in [500, 1000]:
        template = load_config(PROJECT / f'configs/reference_ddim_{budget}_debug.json')
        reference_config = replace(
            template, device='cuda', noise_path=manifest['config']['noise_path'],
            noise_seed=50000, output_dir=f'results/{STUDY_NAME}/reference_ddim_{budget}',
        )
        path = CONFIG_DIR / f'reference_ddim_{budget}.json'
        write_json(path, reference_config.to_dict())
        cli('run', '--config', path)
        references.append(PROJECT / reference_config.output_dir)
    cli('compare', '--runs', references[0], '--reference', references[1],
        '--output', STUDY_DIR / 'reference_convergence.json')
    cli('compare', '--runs', *pilot_runs, '--reference', references[1],
        '--output', STUDY_DIR / 'pilot_mse.json')
else:
    print('Длинный эталон отключён. Основное сравнение FID доступно выше.')

## 10. Сохранить итоговый отчёт

Архив отчёта содержит comparison.json, CSV, сравнительную сетку, проверку реализаций,
pilot, конфигурации и логи, а также отдельную карточку библиотечного baseline.
Полные результаты генерации находятся в трёх скачанных *_full_final_fid_*.zip.
Скачайте также выполненный ноутбук через **Файл → Скачать → Скачать IPYNB**.

In [ ]:
write_json(STUDY_DIR / 'baseline_context.json', {
    'summary': baseline['summary'], 'config': baseline['manifest']['config'],
    'note': 'Separate library baseline protocol. Rank the three common-protocol runs in comparison.json.',
})
report_archive = EXPORT_DIR / f'{STUDY_NAME}_report_{uuid4().hex[:8]}.zip'
with ZipFile(report_archive, 'w', compression=ZIP_DEFLATED) as archive:
    for path in sorted(STUDY_DIR.glob('*')):
        if path.is_file():
            archive.write(path, Path('report') / path.name)
    for path in sorted(CONFIG_DIR.glob('*.json')):
        archive.write(path, Path('configs') / path.name)
    for path in sorted(EXPORT_DIR.glob('*.log')):
        archive.write(path, Path('logs') / path.name)
    archive.write(SOURCE_BUNDLE, 'source.zip')
print('Отчёт готов', report_archive)

In [ ]:
download_one(report_archive)

## Ожидаемый итог и дальнейшее исследование

Для каждого метода получены 5000 изображений, NFE=39, конечный FID, время,
точные шумы, тензоры и полный ZIP. Есть сравнительная таблица и сетка одинаковых шумов.
Численные значения новых методов появятся после фактического запуска на T4.

Далее добавляем бюджеты 19/79, сравниваем uniform_t/log_sigma/karras, подбираем
опорные уровни на calibration и проверяем их на независимых шумах и другом сэмплере.
Алгоритм поиска расписания требует отдельной реализации.